# TTM-Small-4M — T4 A2 | Huấn luyện 1.024 bước liên tục

**Colab Free GPU ($0)**. Bấm **Runtime → Run all**. Một lượt TRAIN 1.024 bước, **không dừng ở 256**; checkpoint tự động ở 256, 512, 768, 1.024. Có log mỗi 32 bước. Chỉ dùng TRAIN và VAL, tuyệt đối không đọc TEST.

Nếu phiên Colab ngắt: chạy lại **chính notebook này**, nó xác minh checkpoint READY và tiếp tục từ mốc gần nhất trong **cùng RunID**. Nếu đã hoàn tất bước 1.024, không chạy lại. Nguồn mã được tải từ đúng commit GitHub và đối chiếu SHA-256. Số 55/90 phút cũ **không còn là điều kiện dừng**, nhưng Colab có thể tự thu hồi GPU. Không dùng Colab trả phí.

Thư mục dữ liệu Drive đã xác minh: `MyDrive/TTM/ttm-small-pilot/t4_private_pack_20261108_v1`. Kết quả mới: `MyDrive/TTM/ttm-small-pilot/t4_science_4m_1024_a2_20261108_v1`.

**Chỉ chạy sau khi muốn bắt đầu lượt khoa học 1.024 bước này.** Khi chạy, bước đầu tiên ghi khóa nghiên cứu một lần vào thư mục Drive riêng tư; không thêm vòng xác nhận kỹ thuật.


In [ ]:
# Ô 1 — tải chính xác các mã đã khóa; không đọc dữ liệu, không dùng GPU
from pathlib import Path
import hashlib, urllib.request, sys, os
ROOT=Path('/content/ttm_t4_a2_1024_exact')
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_COMMIT='314b085c79d2dbb50e2101a29ee59290a70cd586'
PREFIX='https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/'+SOURCE_COMMIT+'/'
FILES={
'p02_source_exact.py':('tools/pilot_t4/integrated/p02_source_exact.py','e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07'),
't4_capacity_smoke.py':('tools/pilot_t4/integrated/t4_capacity_smoke.py','5030855baad73582f897d3cd128ce24621adb39513bf20f527d875e08ea71411'),
't4_eval.py':('tools/pilot_t4/integrated/t4_eval.py','1502ea5773ab0f7c0869097ecb19b610867d193728d7ce3a78ef82439767ab5b'),
't4_capacity_accept_a2.py':('tools/pilot_t4/a2_continuous1024/t4_capacity_accept_a2.py','8db7ce9111744523a6c2e558a50677e19c56788298a0619eb4cfa924ea6a03c2'),
't4_train_science_a2.py':('tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py','c6298f3381efd4b54e4b3e7aeb8143354f36bb1ed9fd87d08900cee18f7f90e6')}
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
for name,(rel,expected) in FILES.items():
    dst=ROOT/name
    if dst.exists():
        assert sha256_file(dst)==expected, 'SOURCE_EXISTING_MISMATCH:'+name
    else:
        with urllib.request.urlopen(PREFIX+rel,timeout=45) as response:
            body=response.read()
        assert hashlib.sha256(body).hexdigest()==expected,'SOURCE_SHA_MISMATCH:'+name
        with dst.open('xb') as f: f.write(body)
    print('T4_A2_SOURCE_PIN=PASS',name)
sys.path.insert(0,str(ROOT))
print('T4_A2_SOURCE_COMMIT='+SOURCE_COMMIT)


In [ ]:
# Ô 2 — xác minh chính xác TRAIN/VAL riêng tư, tạo lock đúng một lượt, không dùng TEST
from google.colab import drive
from pathlib import Path
import os, json, hashlib, torch, shutil
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'COLAB_FREE_GPU_REQUIRED'
print('T4_GPU='+torch.cuda.get_device_name(0), 'TORCH='+str(torch.__version__))
BASE=Path('/content/drive/MyDrive/TTM/ttm-small-pilot')
PRIVATE=BASE/'t4_private_pack_20261108_v1'
CAPACITY=BASE/'t4_capacity_synthetic_20261108_v1'/'T4_CAPACITY_REPORT.json'
SRC=PRIVATE/'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz'
MNF=PRIVATE/'T4_PRIVATE_PACK_MANIFEST.json'
RUN=BASE/'t4_science_4m_1024_a2_20261108_v1'
DATA_SHA='184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913'
MANIFEST_SHA='ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650'
CAPACITY_SHA='c55a568496a888bc3abc95c0231a5ed17d3a6f366361571b6f89a4457d09f1ee'
assert SRC.is_file() and MNF.is_file() and CAPACITY.is_file(), 'MISSING_PINNED_DATA_OR_CAPACITY'
assert SRC.stat().st_size==1186107 and sha256_file(SRC)==DATA_SHA,'PRIVATE_NPZ_SHA_FAILED'
assert sha256_file(MNF)==MANIFEST_SHA,'PRIVATE_MANIFEST_SHA_FAILED'
assert sha256_file(CAPACITY)==CAPACITY_SHA,'CAPACITY_SHA_FAILED'
manifest=json.loads(MNF.read_text('utf8'))
assert manifest['npz_sha256']==DATA_SHA and manifest['contains_test'] is False
assert manifest['counts']['train_stories']==2048 and manifest['counts']['val_stories']==256
assert manifest['counts']['test_stories_tokenized']==0
assert manifest['counts']['train_next_token_targets_unique']==498287
capacity=json.loads(CAPACITY.read_text('utf8'))
assert capacity['science_executed'] is False and capacity['status']=='PASS_PROVISIONAL_CAPACITY_ONLY'
from t4_capacity_accept_a2 import assess
review=assess(capacity,manifest)
review['capacity_sha256']=CAPACITY_SHA
review['pack_manifest_sha256']=MANIFEST_SHA
selected=review['selected']
assert review['status']=='CAPACITY_PASS_PROVISIONAL_FOR_SCIENCE_REVIEW'
assert selected['precision']=='fp16' and selected['microbatch_windows']==8
def write_exact_once(path,obj):
    raw=(json.dumps(obj,indent=2,sort_keys=True)+'\n').encode('utf-8')
    if path.exists():
        assert path.read_bytes()==raw, 'LOCK_OR_REVIEW_CONFLICT:'+path.name
    else:
        with path.open('xb') as f:
            f.write(raw);f.flush();os.fsync(f.fileno())
    return sha256_file(path)
REVIEW=PRIVATE/'T4_CAPACITY_PRIVATE_PACK_REVIEW_A2.json'
review_sha=write_exact_once(REVIEW,review)
LOCK=PRIVATE/'T4_SCIENCE_LOCK_A2_1024.json'
LOCK_OBJ={
 'schema':'ttm.t4.a2.science.lock.v1',
 'authorization':'ONE_SHOT_T4_SCIENCE_GRANTED',
 'science_seed':20261108,
 'budget':{'optimizer_steps':1024,'target_tokens_per_step':4096,
           'time_policy':'COLAB_FREE_RUN_UNTIL_1024_OR_PREEMPTED'},
 'train_pack_sha256':DATA_SHA,
 'capacity_report_sha256':CAPACITY_SHA,
 'capacity_review_sha256':review_sha,
 'precision':'fp16','microbatch_windows':8,
 'p02_git_blob':'9762343e06cf800b50574c2871ae1ed58537fb18',
 'trainer_a2_sha256':FILES['t4_train_science_a2.py'][1],
 'run_id':RUN.name,'no_test_access':True,
 'h1_primary_checkpoint_step':1024}
LOCK_SHA=write_exact_once(LOCK,LOCK_OBJ)
print('T4_PRIVATE_DRIVE_SHA=PASS_EXACT')
print('T4_SCIENCE_LOCK_SHA256='+LOCK_SHA)
print('T4_PLAN=CONTINUOUS_1024_NO_STOP_AT_256')


In [ ]:
# Ô 3 — chạy một lượt đến 1.024; tự checkpoint, resume khi cần và luôn flush Drive
import subprocess, sys, json, shutil, os
local_npz=ROOT/'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz'
local_manifest=ROOT/'T4_PRIVATE_PACK_MANIFEST.json'
if local_npz.exists(): assert sha256_file(local_npz)==DATA_SHA,'LOCAL_DATA_MISMATCH'
else: shutil.copyfile(SRC,local_npz)
if local_manifest.exists(): assert sha256_file(local_manifest)==MANIFEST_SHA,'LOCAL_MANIFEST_MISMATCH'
else: shutil.copyfile(MNF,local_manifest)
assert sha256_file(local_npz)==DATA_SHA and sha256_file(local_manifest)==MANIFEST_SHA
assert sha256_file(LOCK)==LOCK_SHA and sha256_file(REVIEW)==review_sha
resume=None
completed=False
if RUN.exists():
    if (RUN/'T4_FINAL_UNADJUDICATED.json').is_file():
        completed=True
        print('T4_ALREADY_COMPLETED_1024=NO_RERUN')
    else:
        croot=RUN/'checkpoints'
        folders=sorted(croot.glob('step_*')) if croot.exists() else []
        if not folders: raise RuntimeError('RUN_EXISTS_WITHOUT_READY_CHECKPOINT_STOP')
        if any(not (f/'READY.json').is_file() for f in folders):
            raise RuntimeError('PARTIAL_CHECKPOINT_NEEDS_REVIEW_NO_AUTO_RERUN')
        resume=folders[-1]
        assert resume.name in ('step_0256','step_0512','step_0768'), 'UNEXPECTED_RESUME_STEP'
        print('T4_RESUME_SAME_RUN_FROM='+resume.name)
if not completed:
    args=[sys.executable,'-u',str(ROOT/'t4_train_science_a2.py'),'--mode','science',
      '--p02',str(ROOT/'p02_source_exact.py'),
      '--data',str(local_npz),'--data-manifest',str(local_manifest),
      '--capacity-report',str(CAPACITY),'--capacity-review',str(REVIEW),
      '--drive-run',str(RUN),'--lock-json',str(LOCK),'--lock-sha256',LOCK_SHA]
    if resume is not None: args.extend(['--resume-checkpoint-dir',str(resume)])
    try:
        with subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as proc:
            for line in proc.stdout: print(line,end='',flush=True)
            rc=proc.wait()
        if rc!=0: raise RuntimeError('T4_STOP_KEEP_DRIVE_EVIDENCE_NO_NEW_RUN_EXIT_'+str(rc))
        final=json.loads((RUN/'T4_FINAL_UNADJUDICATED.json').read_text('utf8'))
        assert final['last_step']==1024 and final['exact_scored_training_tokens']==4194304
        print('T4_A2_COMPLETE_1024=PASS_UNADJUDICATED')
        print('T4_A2_H1_VAL_RESULT='+json.dumps(final['h1_val'],sort_keys=True))
    finally:
        try: drive.flush_and_unmount()
        finally: print('T4_DRIVE_FLUSH_ATTEMPTED_READBACK_REQUIRED')
else:
    print('T4_FINAL_REPORT_EXISTS=DO_NOT_RETRAIN')
    drive.flush_and_unmount()
